# OrderBot

We can automate the collection of user prompts and assistant responses to build an  OrderBot. The OrderBot will take orders at a pizza restaurant. 

We need to install panel to run this app properly. In addition, directly install panel within the notebook will not work. Below are the steps to follow:

- go to the terminal, activate AI500Env by typing conda activate AI500Env

- on the next line, type pip install panel

In [ ]:
# We cannot run below line to install panel in your local environment. It will not work for this environment

#!pip install panel

In [ ]:
from dotenv import load_dotenv
import openai
from openai import OpenAI
import os

# define the environment path. it is at the parent folder
env_path ="../.env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

# select a model.
# You can try different models to see which one works the best.

my_model="gpt-4o-mini"

def get_completion(prompt, model=my_model):
    messages = [{"role": "user", "content": prompt}]
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # we want the model to give us the best response, no variation.
    )
    return response.choices[0].message.content

def get_completion_from_messages(messages, model=my_model, temperature=0):
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))
 
    return pn.Column(*panels)

# Define function to quit the application
def quit_application(_):
    dashboard.clear()  # Clear the dashboard
    quit_message = pn.pane.Markdown(
        "Thank you for using the OrderBot. The application has been closed.", 
        width=600, 
        style={'color': 'red', 'font-size': '16px'}
    )
    panels.append(quit_message)  # Show quit message
    pn.io.server.stop()  # St

import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant.
You first greet the customer, then collects the order, 
and then asks if it's a pickup or delivery.
You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else.
If it's a delivery, you ask for an address.
tell customer the final price of the order at the end, please make sure you add cost for extra toppings.
if it is a delivery, added 5 dollor delivery fee.
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely
identify the item from the menu.
You respond in a short, very conversational friendly style.
The menu includes
pepperoni pizza  12.95, 10.00, 7.00
cheese pizza   10.95, 9.25, 6.50
eggplant pizza   11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")
button_quit = pn.widgets.Button(name="Quit")

interactive_conversation = pn.bind(collect_messages, button_conversation)
button_quit.on_click(quit_application)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation, button_quit),
    pn.panel(interactive_conversation, loading_indicator=True, height=600),
)

dashboard

In [ ]:
context

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)

response = get_completion_from_messages(messages, temperature=0)
print(response)